In [2]:
from pathlib import Path
# from gaze_av_aloha.robot.env_no_left import RealEnv
# from gaze_av_aloha.robot.config import REAL_DT, FPS
from tqdm import tqdm
import einops
import numpy as np
import torch
import imageio
import os
from torch import Tensor
import torchvision.transforms as v2
import time
import safetensors.torch
from diffusers.training_utils import EMAModel
from gaze_av_aloha.policies.gaze_policy.gaze_policy import GazePolicy
import cv2
from torch.utils.data import DataLoader, Subset
from gym_av_aloha.datasets.av_aloha_dataset import AVAlohaDataset
from gaze_av_aloha.policies.gaze_policy.gaze_model import GazeModel
import torch
from torch import nn, Tensor
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import imageio
import cv2
from IPython.display import Video, display
import einops
import kornia.augmentation as K
import torch.nn.utils as nn_utils
import os
from hydra import initialize_config_dir, compose
from gaze_av_aloha.configs import Config
from torchvision.transforms import Resize, Normalize
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import RandomApply

In [3]:
REAL_DT = 0.03
FPS = round(1 / REAL_DT)
image_keys = ["observation.images.left_eye_cam", "observation.images.right_eye_cam"]
eye_keys = ["left_eye", "right_eye"]
state=["observation.state"]
dataset = "Jinyu220/coin_2"

delta_timestamps = {
    k: [0] for k in image_keys + eye_keys+state
}
print(delta_timestamps)
dataset = AVAlohaDataset(
    repo_id=dataset,
    delta_timestamps=delta_timestamps,
)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
)
ranges = [
    [0, 20],
]
eval_dataset = Subset(dataset, sum([list(range(start, end)) for start, end in ranges], []))
eval_dataloader = DataLoader(eval_dataset, batch_size=1, shuffle=False)
img = None 
video = []


{'observation.images.left_eye_cam': [0], 'observation.images.right_eye_cam': [0], 'left_eye': [0], 'right_eye': [0], 'observation.state': [0]}
avalohadataset: /home/jinyu/GitHub/gaze-av-aloha/gym_av_aloha/outputs/Jinyu220/coin_2


In [ ]:
transforms_augmentation = transforms.Compose([
    #transforms.ToPILImage(), 

    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),

    transforms.GaussianBlur(kernel_size=3),
    #RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.5),
    #transforms.ToTensor(),
  
])


image_transforms_augmentation = transforms.Compose([
    transforms.ToPILImage(),
    RandomApply([transforms_augmentation], p=0.7),  
    transforms.ToTensor(),
])
